In [2]:
import numpy as np
import pandas as pd
from scipy import stats
import os

In [3]:

PROCESSED_DIR = r"C:\Users\ZBOOK\OneDrive\Documents\Olist E Commerce Analytics Sales, Logistics & Customer Satisfaction Dashboard\Dataset\processed"

fact_orders = pd.read_csv(os.path.join(PROCESSED_DIR, "fact_orders.csv"))
fact_reviews = pd.read_csv(os.path.join(PROCESSED_DIR, "fact_reviews.csv"))

orders_reviews = fact_orders.merge(fact_reviews[['order_id', 'review_score']], on='order_id', how='inner')

print(fact_orders.shape, fact_reviews.shape, orders_reviews.shape)

(98822, 16) (97839, 5) (97839, 17)


In [4]:
order_value = fact_orders['order_total'].dropna()

print(f"Mean:   R${order_value.mean():,.2f}")
print(f"Median: R${order_value.median():,.2f}")
print(f"Std:    R${order_value.std():,.2f}")
print(f"25th percentile: R${order_value.quantile(0.25):,.2f}")
print(f"50th percentile: R${order_value.quantile(0.50):,.2f}")
print(f"75th percentile: R${order_value.quantile(0.75):,.2f}")
print(f"90th percentile: R${order_value.quantile(0.90):,.2f}")
print(f"99th percentile: R${order_value.quantile(0.99):,.2f}")

Mean:   R$160.25
Median: R$105.28
Std:    R$219.13
25th percentile: R$61.96
50th percentile: R$105.28
75th percentile: R$176.74
90th percentile: R$307.14
99th percentile: R$1,055.74


In [5]:
delivered = orders_reviews[orders_reviews['order_delivered_customer_date'].notna()].copy()
delivered['order_purchase_timestamp'] = pd.to_datetime(delivered['order_purchase_timestamp'])
delivered['order_delivered_customer_date'] = pd.to_datetime(delivered['order_delivered_customer_date'])
delivered['delivery_days'] = (delivered['order_delivered_customer_date'] - delivered['order_purchase_timestamp']).dt.days

pearson_r, pearson_p = stats.pearsonr(delivered['delivery_days'], delivered['review_score'])
spearman_r, spearman_p = stats.spearmanr(delivered['delivery_days'], delivered['review_score'])

print(f"Pearson r:  {pearson_r:.4f}, p-value: {pearson_p:.2e}")
print(f"Spearman r: {spearman_r:.4f}, p-value: {spearman_p:.2e}")

Pearson r:  -0.3342, p-value: 0.00e+00
Spearman r: -0.2345, p-value: 0.00e+00


In [6]:
delivered['is_late'] = delivered['delivery_delay_days'] > 0

on_time_scores = delivered[delivered['is_late'] == False]['review_score']
late_scores = delivered[delivered['is_late'] == True]['review_score']

print(f"On-time mean review score: {on_time_scores.mean():.2f} (n={len(on_time_scores)})")
print(f"Late mean review score:    {late_scores.mean():.2f} (n={len(late_scores)})")

levene_stat, levene_p = stats.levene(on_time_scores, late_scores)
print(f"\nLevene's test p-value: {levene_p:.4f}")

if levene_p < 0.05:
    t_stat, t_p = stats.ttest_ind(on_time_scores, late_scores, equal_var=False)
    print("Using Welch's t-test (unequal variances)")
else:
    t_stat, t_p = stats.ttest_ind(on_time_scores, late_scores, equal_var=True)
    print("Using standard t-test (equal variances)")

print(f"t-statistic: {t_stat:.4f}, p-value: {t_p:.2e}")

u_stat, u_p = stats.mannwhitneyu(on_time_scores, late_scores, alternative='two-sided')
print(f"\nMann-Whitney U: {u_stat:.2f}, p-value: {u_p:.2e}")


On-time mean review score: 4.29 (n=89245)
Late mean review score:    2.27 (n=6362)

Levene's test p-value: 0.0000
Using Welch's t-test (unequal variances)
t-statistic: 100.7804, p-value: 0.00e+00

Mann-Whitney U: 465237173.50, p-value: 0.00e+00


In [8]:
mean_val = order_value.mean()
std_err = stats.sem(order_value) 
ci_low, ci_high = stats.t.interval(0.95, len(order_value)-1, loc=mean_val, scale=std_err)

print(f"Mean order value: R${mean_val:,.2f}")
print(f"95% CI: R${ci_low:,.2f} to R${ci_high:,.2f}")

Mean order value: R$160.25
95% CI: R$158.88 to R$161.62


In [9]:
fact_payments = pd.read_csv(os.path.join(PROCESSED_DIR, "fact_payments.csv"))
orders_payments_reviews = fact_orders.merge(fact_payments[['order_id', 'payment_type']], on='order_id', how='inner') \
                                       .merge(fact_reviews[['order_id', 'review_score']], on='order_id', how='inner')

orders_payments_reviews['review_category'] = pd.cut(
    orders_payments_reviews['review_score'],
    bins=[0, 2, 3, 5],
    labels=['Negative (1-2)', 'Neutral (3)', 'Positive (4-5)']
)

contingency_table = pd.crosstab(orders_payments_reviews['payment_type'], orders_payments_reviews['review_category'])
print(contingency_table)

chi2, p_value, dof, expected = stats.chi2_contingency(contingency_table)
print(f"\nChi-square statistic: {chi2:.2f}")
print(f"p-value: {p_value:.2e}")
print(f"Degrees of freedom: {dof}")

review_category  Negative (1-2)  Neutral (3)  Positive (4-5)
payment_type                                                
boleto                     2744         1671           15088
credit_card               10819         6139           58622
debit_card                  190          117            1207
voucher                     891          485            4246

Chi-square statistic: 22.48
p-value: 9.89e-04
Degrees of freedom: 6


In [10]:
fact_order_items = pd.read_csv(os.path.join(PROCESSED_DIR, "fact_order_items.csv"))
dim_products = pd.read_csv(os.path.join(PROCESSED_DIR, "dim_products.csv"))

items_products = fact_order_items.merge(dim_products[['product_id', 'product_category_name_english']], on='product_id')
items_products_reviews = items_products.merge(fact_reviews[['order_id', 'review_score']], on='order_id')

top5_categories = items_products_reviews['product_category_name_english'].value_counts().head(5).index.tolist()
print(f"Top 5 categories: {top5_categories}")

groups = [
    items_products_reviews[items_products_reviews['product_category_name_english'] == cat]['review_score']
    for cat in top5_categories
]

f_stat, p_value = stats.f_oneway(*groups)
print(f"\nANOVA F-statistic: {f_stat:.4f}")
print(f"p-value: {p_value:.2e}")

for cat, group in zip(top5_categories, groups):
    print(f"{cat}: mean = {group.mean():.2f}, n = {len(group)}")

Top 5 categories: ['bed_bath_table', 'health_beauty', 'sports_leisure', 'furniture_decor', 'computers_accessories']

ANOVA F-statistic: 65.5414
p-value: 2.27e-55
bed_bath_table: mean = 3.90, n = 10936
health_beauty: mean = 4.15, n = 9533
sports_leisure: mean = 4.12, n = 8504
furniture_decor: mean = 3.93, n = 8178
computers_accessories: mean = 3.95, n = 7714


In [11]:
n = contingency_table.sum().sum()
min_dim = min(contingency_table.shape) - 1
cramers_v = np.sqrt(chi2 / (n * min_dim))
print(f"Cramér's V: {cramers_v:.4f}")

Cramér's V: 0.0105
